# 12. Interactive Gradio Demo (Model Interface)

This section provides an interactive interface for testing the trained **ResNet18** and **EfficientNet-B0** models on uploaded brain MRI images.

The demo allows the user to:

- Run **ResNet18** individually.
- Run **EfficientNet-B0** individually.
- Compare both models on the same MRI image.
- View the predicted class and model confidence.
- Inspect class probabilities for all four categories.
- Generate **Grad-CAM** heatmaps to visualize which image regions influenced each model's prediction.
- **Flag difficult or incorrectly classified cases for later review.**

The interface uses the previously saved trained checkpoints:

- `best_resnet18.pth`
- `best_efficientnet_b0.pth`

**No model retraining is performed in this section.**

---

## Flagged Cases for Later Review

The flagging feature is used to save interesting or incorrectly classified MRI cases for later error analysis.

The image name is entered **after the prediction**, so the true class information is **not provided to the models** and does not influence their prediction.

### Flagging Workflow

```text
Upload MRI image
      ↓
Analyze MRI
      ↓
Models make prediction
without seeing the image name / true class
      ↓
Prediction + Confidence + Grad-CAM
      ↓
Enter Image Name / ID
Example: Te-gl_59
      ↓
True Class is identified from the image ID
Example: Glioma
      ↓
🚩 Flag for Review
      ↓
Save case for later analysis
      ↓
flagged_cases/
└── Te-gl_59__2026-09-20_15-24-11/
    ├── original_mri.png
    ├── resnet_gradcam.png
    ├── efficientnet_gradcam.png
    └── case_info.txt

In [2]:
# ============================================================
# Cell 45: Interactive Gradio Demo
# ============================================================

# ============================================================
# Brain Tumor MRI Classification — Gradio Demo
# ResNet18 / EfficientNet-B0 / Compare Both
# Prediction + Confidence + Probabilities + Grad-CAM
# Flagged Cases for Later Review
# NO TRAINING
# ============================================================

import sys
import subprocess
from pathlib import Path
from datetime import datetime
import re

import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image


# ============================================================
# 1. Install / Import Gradio
# ============================================================

try:
    import gradio as gr
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "gradio"]
    )
    import gradio as gr


# ============================================================
# 2. Install / Import Grad-CAM
# ============================================================

try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

except ImportError:

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "grad-cam"]
    )

    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget


# ============================================================
# 3. Device
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", DEVICE)


# ============================================================
# 4. Classes
# ============================================================

CLASS_NAMES = [
    "glioma",
    "meningioma",
    "notumor",
    "pituitary"
]

DISPLAY_NAMES = {
    "glioma": "Glioma",
    "meningioma": "Meningioma",
    "notumor": "No Tumor",
    "pituitary": "Pituitary"
}


# ============================================================
# 5. Checkpoints
# ============================================================

RESNET_PATH = Path("Models/best_resnet18.pth")
EFFICIENTNET_PATH = Path("Models/best_efficientnet_b0.pth")

if not RESNET_PATH.exists():
    raise FileNotFoundError(
        f"{RESNET_PATH} was not found."
    )

if not EFFICIENTNET_PATH.exists():
    raise FileNotFoundError(
        f"{EFFICIENTNET_PATH} was not found."
    )


# ============================================================
# 6. Load checkpoint helper
# ============================================================

def load_checkpoint(path):

    try:

        return torch.load(
            path,
            map_location=DEVICE,
            weights_only=True
        )

    except TypeError:

        return torch.load(
            path,
            map_location=DEVICE
        )


# ============================================================
# 7. Load ResNet18
# ============================================================

resnet_model = models.resnet18(
    weights=None
)

num_features = (
    resnet_model.fc.in_features
)

resnet_model.fc = nn.Linear(
    num_features,
    len(CLASS_NAMES)
)

resnet_model.load_state_dict(
    load_checkpoint(
        RESNET_PATH
    )
)

resnet_model = (
    resnet_model
    .to(DEVICE)
)

resnet_model.eval()


# ============================================================
# 8. Load EfficientNet-B0
# ============================================================

efficientnet_model = models.efficientnet_b0(
    weights=None
)

in_features = (
    efficientnet_model
    .classifier[1]
    .in_features
)

efficientnet_model.classifier[1] = nn.Linear(
    in_features,
    len(CLASS_NAMES)
)

efficientnet_model.load_state_dict(
    load_checkpoint(
        EFFICIENTNET_PATH
    )
)

efficientnet_model = (
    efficientnet_model
    .to(DEVICE)
)

efficientnet_model.eval()


print(
    "ResNet18 loaded successfully."
)

print(
    "EfficientNet-B0 loaded successfully."
)


# ============================================================
# 9. Preprocessing
# ============================================================

inference_transform = transforms.Compose([

    transforms.Lambda(
        lambda img:
        img.convert("RGB")
    ),

    transforms.Resize(
        (224, 224)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],

        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


# ============================================================
# 10. Prepare image
# ============================================================

def prepare_image(image):

    image = image.convert(
        "RGB"
    )

    tensor = inference_transform(
        image
    )

    input_tensor = (
        tensor
        .unsqueeze(0)
        .to(DEVICE)
    )

    return (
        image,
        input_tensor
    )


# ============================================================
# 11. Prediction
# ============================================================

def run_prediction(
    model,
    input_tensor
):

    model.eval()

    with torch.no_grad():

        outputs = model(
            input_tensor
        )

        probabilities = torch.softmax(
            outputs,
            dim=1
        )[0]

        predicted_index = int(
            torch.argmax(
                probabilities
            ).item()
        )

    predicted_class = (
        CLASS_NAMES[
            predicted_index
        ]
    )

    confidence = float(
        probabilities[
            predicted_index
        ]
        .cpu()
        .item()
    )

    probability_dict = {

        DISPLAY_NAMES[class_name]:
            float(
                probabilities[i]
                .cpu()
                .item()
            )

        for i, class_name
        in enumerate(CLASS_NAMES)
    }

    return (
        predicted_index,
        predicted_class,
        confidence,
        probability_dict
    )


# ============================================================
# 12. Grad-CAM
# ============================================================

def generate_gradcam(
    model,
    model_name,
    original_image,
    input_tensor,
    predicted_index
):

    if model_name == "ResNet18":

        target_layers = [
            model.layer4[-1]
        ]

    elif model_name == "EfficientNet-B0":

        target_layers = [
            model.features[-1]
        ]

    else:

        raise ValueError(
            "Unknown model."
        )

    cam = GradCAM(
        model=model,
        target_layers=target_layers
    )

    targets = [
        ClassifierOutputTarget(
            predicted_index
        )
    ]

    grayscale_cam = cam(
        input_tensor=input_tensor,
        targets=targets
    )[0]

    display_image = (
        original_image
        .resize(
            (224, 224)
        )
    )

    rgb_image = (
        np.array(
            display_image
        )
        .astype(
            np.float32
        )
        / 255.0
    )

    visualization = show_cam_on_image(
        rgb_image,
        grayscale_cam,
        use_rgb=True
    )

    return visualization


# ============================================================
# 13. Analyze one model
# ============================================================

def analyze_single_model(
    model,
    model_name,
    original_image,
    input_tensor
):

    (
        predicted_index,
        predicted_class,
        confidence,
        probabilities

    ) = run_prediction(
        model,
        input_tensor
    )

    gradcam_image = generate_gradcam(

        model=model,

        model_name=model_name,

        original_image=original_image,

        input_tensor=input_tensor,

        predicted_index=predicted_index
    )

    prediction_text = (

        f"Prediction: "
        f"{DISPLAY_NAMES[predicted_class]}\n"

        f"Confidence: "
        f"{confidence * 100:.2f}%"
    )

    return {

        "class":
            predicted_class,

        "display_class":
            DISPLAY_NAMES[
                predicted_class
            ],

        "confidence":
            confidence,

        "text":
            prediction_text,

        "probabilities":
            probabilities,

        "gradcam":
            gradcam_image
    }


# ============================================================
# 14. Main analysis
# ============================================================

def analyze_mri(
    image,
    mode
):

    if image is None:

        raise gr.Error(
            "Please upload an MRI image first."
        )

    (
        original_image,
        input_tensor

    ) = prepare_image(
        image
    )


    # --------------------------------------------------------
    # ResNet18 only
    # --------------------------------------------------------

    if mode == "ResNet18":

        result = analyze_single_model(

            resnet_model,

            "ResNet18",

            original_image,

            input_tensor
        )

        return (

            result["text"],

            result["probabilities"],

            result["gradcam"],

            "",

            {},

            None,

            ""
        )


    # --------------------------------------------------------
    # EfficientNet-B0 only
    # --------------------------------------------------------

    elif mode == "EfficientNet-B0":

        result = analyze_single_model(

            efficientnet_model,

            "EfficientNet-B0",

            original_image,

            input_tensor
        )

        return (

            "",

            {},

            None,

            result["text"],

            result["probabilities"],

            result["gradcam"],

            ""
        )


    # --------------------------------------------------------
    # Compare both
    # --------------------------------------------------------

    else:

        resnet_result = analyze_single_model(

            resnet_model,

            "ResNet18",

            original_image,

            input_tensor
        )

        efficientnet_result = analyze_single_model(

            efficientnet_model,

            "EfficientNet-B0",

            original_image,

            input_tensor
        )

        if (
            resnet_result["class"]
            ==
            efficientnet_result["class"]
        ):

            comparison_text = (

                "Models agree.\n\n"

                f"Prediction: "
                f"{resnet_result['display_class']}\n\n"

                f"ResNet18 confidence: "
                f"{resnet_result['confidence'] * 100:.2f}%\n"

                f"EfficientNet-B0 confidence: "
                f"{efficientnet_result['confidence'] * 100:.2f}%"
            )

        else:

            comparison_text = (

                "Models disagree.\n\n"

                f"ResNet18: "
                f"{resnet_result['display_class']} "
                f"({resnet_result['confidence'] * 100:.2f}%)\n\n"

                f"EfficientNet-B0: "
                f"{efficientnet_result['display_class']} "
                f"({efficientnet_result['confidence'] * 100:.2f}%)"
            )

        return (

            resnet_result["text"],

            resnet_result["probabilities"],

            resnet_result["gradcam"],

            efficientnet_result["text"],

            efficientnet_result["probabilities"],

            efficientnet_result["gradcam"],

            comparison_text
        )


# ============================================================
# 15. Visibility
# ============================================================

def update_visibility(mode):

    if mode == "ResNet18":

        return (

            gr.update(
                visible=True
            ),

            gr.update(
                visible=False
            ),

            gr.update(
                visible=False
            )
        )

    elif mode == "EfficientNet-B0":

        return (

            gr.update(
                visible=False
            ),

            gr.update(
                visible=True
            ),

            gr.update(
                visible=False
            )
        )

    else:

        return (

            gr.update(
                visible=True
            ),

            gr.update(
                visible=True
            ),

            gr.update(
                visible=True
            )
        )


# ============================================================
# 16. Flagged Cases — Save for Later Review
# ============================================================

FLAGGED_DIR = Path(
    "flagged_cases"
)

FLAGGED_DIR.mkdir(
    exist_ok=True
)


# ------------------------------------------------------------
# Infer true class from image ID
#
# Example:
# Te-gl_59 -> Glioma
#
# This information is ONLY used when saving a flagged case.
# It is NOT provided to either model.
# ------------------------------------------------------------

def infer_true_class(
    image_id
):

    if not image_id:

        return "Unknown"

    image_id_lower = (
        image_id
        .strip()
        .lower()
    )

    # Split the filename into pieces.
    # Example:
    # Te-gl_59
    # becomes:
    # ["te", "gl", "59"]

    parts = re.split(
        r"[-_\s]+",
        image_id_lower
    )

    class_code_map = {

        "gl":
            "Glioma",

        "glioma":
            "Glioma",

        "me":
            "Meningioma",

        "men":
            "Meningioma",

        "meningioma":
            "Meningioma",

        "no":
            "No Tumor",

        "nt":
            "No Tumor",

        "notumor":
            "No Tumor",

        "no-tumor":
            "No Tumor",

        "pi":
            "Pituitary",

        "pit":
            "Pituitary",

        "pituitary":
            "Pituitary"
    }

    for part in parts:

        if part in class_code_map:

            return class_code_map[
                part
            ]

    return "Unknown"


# ------------------------------------------------------------
# Update true class display when image ID changes
# ------------------------------------------------------------

def update_true_class(
    image_id
):

    return infer_true_class(
        image_id
    )


# ------------------------------------------------------------
# Clean filename so it is safe to use as folder name
# ------------------------------------------------------------

def clean_image_id(
    image_id
):

    if not image_id:

        return "Unnamed_Image"

    cleaned = re.sub(
        r'[<>:"/\\|?*]',
        "_",
        image_id.strip()
    )

    cleaned = cleaned.strip(
        ". "
    )

    if not cleaned:

        return "Unnamed_Image"

    return cleaned


# ------------------------------------------------------------
# Save image helper
# ------------------------------------------------------------

def save_numpy_or_pil_image(
    image_data,
    save_path
):

    if image_data is None:

        return

    if isinstance(
        image_data,
        Image.Image
    ):

        image_data.save(
            save_path
        )

        return

    array = np.asarray(
        image_data
    )

    if array.dtype != np.uint8:

        array = np.clip(
            array,
            0,
            255
        ).astype(
            np.uint8
        )

    Image.fromarray(
        array
    ).save(
        save_path
    )


# ------------------------------------------------------------
# Save flagged case
# ------------------------------------------------------------

def save_flagged_case(

    image,

    image_id,

    mode,

    resnet_prediction,

    resnet_gradcam,

    efficientnet_prediction,

    efficientnet_gradcam,

    comparison_text
):

    if image is None:

        raise gr.Error(
            "Please upload an MRI image first."
        )

    if not image_id or not image_id.strip():

        raise gr.Error(
            "Please enter the image name / ID before flagging."
        )

    if (
        not resnet_prediction
        and
        not efficientnet_prediction
        and
        not comparison_text
    ):

        raise gr.Error(
            "Please analyze the MRI first, then flag it."
        )


    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    timestamp = datetime.now().strftime(
        "%Y-%m-%d_%H-%M-%S"
    )

    clean_id = clean_image_id(
        image_id
    )

    true_class = infer_true_class(
        image_id
    )


    # --------------------------------------------------------
    # Folder name
    #
    # Example:
    # Te-gl_59__2026-09-20_15-24-11
    # --------------------------------------------------------

    folder_name = (
        f"{clean_id}"
        f"__"
        f"{timestamp}"
    )

    case_dir = (
        FLAGGED_DIR
        /
        folder_name
    )

    case_dir.mkdir(
        parents=True,
        exist_ok=False
    )


    # --------------------------------------------------------
    # Save original MRI
    # --------------------------------------------------------

    original_image = image.convert(
        "RGB"
    )

    original_image.save(
        case_dir
        /
        "original_mri.png"
    )


    # --------------------------------------------------------
    # Save ResNet18 Grad-CAM if available
    # --------------------------------------------------------

    if resnet_gradcam is not None:

        save_numpy_or_pil_image(

            resnet_gradcam,

            case_dir
            /
            "resnet_gradcam.png"
        )


    # --------------------------------------------------------
    # Save EfficientNet-B0 Grad-CAM if available
    # --------------------------------------------------------

    if efficientnet_gradcam is not None:

        save_numpy_or_pil_image(

            efficientnet_gradcam,

            case_dir
            /
            "efficientnet_gradcam.png"
        )


    # --------------------------------------------------------
    # Save case information
    # --------------------------------------------------------

    info_text = f"""Flagged MRI Case
================

Image ID
--------
{image_id}

True Class
----------
{true_class}

Flagged At
----------
{timestamp}

View Mode
---------
{mode}


ResNet18
--------
{resnet_prediction if resnet_prediction else "Not used"}


EfficientNet-B0
---------------
{efficientnet_prediction if efficientnet_prediction else "Not used"}


Model Comparison
----------------
{comparison_text if comparison_text else "Not available"}
"""


    with open(

        case_dir
        /
        "case_info.txt",

        "w",

        encoding="utf-8"

    ) as file:

        file.write(
            info_text
        )


    # --------------------------------------------------------
    # Confirmation message
    # --------------------------------------------------------

    return (

        f"Saved for review\n\n"

        f"Image ID: {image_id}\n"

        f"True Class: {true_class}\n"

        f"Folder: {folder_name}"
    )


# ============================================================
# 17. Gradio UI
# ============================================================

with gr.Blocks(
    title="Brain Tumor MRI Classification"
) as demo:


    gr.Markdown(
        """
# Brain Tumor MRI Classification

Interactive educational demo using trained **ResNet18**
and **EfficientNet-B0** models.

The application displays model predictions, confidence,
class probabilities, and Grad-CAM explainability.

Flagged cases can be saved locally for later review.

**For educational and research purposes only.
Not intended for medical diagnosis.**
        """
    )


    # --------------------------------------------------------
    # Input
    # --------------------------------------------------------

    with gr.Row():

        with gr.Column():

            image_input = gr.Image(
                type="pil",
                label="Upload Brain MRI"
            )


            model_mode = gr.Radio(

                choices=[
                    "ResNet18",
                    "EfficientNet-B0",
                    "Compare Both"
                ],

                value="Compare Both",

                label="View Mode"
            )


            analyze_button = gr.Button(

                "Analyze MRI",

                variant="primary"
            )


            gr.Markdown(
                "### Flag for Later Review"
            )


            image_id_input = gr.Textbox(

                label="Image Name / ID",

                placeholder="Example: Te-gl_59"
            )


            true_class_display = gr.Textbox(

                label="Detected True Class",

                value="Unknown",

                interactive=False
            )


            flag_button = gr.Button(

                "🚩 Flag for Review",

                variant="secondary"
            )


            flag_status = gr.Textbox(

                label="Flag Status",

                lines=4,

                interactive=False
            )


    # --------------------------------------------------------
    # Comparison
    # --------------------------------------------------------

    with gr.Column(
        visible=True
    ) as comparison_section:


        gr.Markdown(
            "## Model Comparison"
        )


        comparison_output = gr.Textbox(

            label="Comparison",

            lines=6
        )


    # --------------------------------------------------------
    # ResNet18
    # --------------------------------------------------------

    with gr.Column(
        visible=True
    ) as resnet_section:


        gr.Markdown(
            "## ResNet18"
        )


        with gr.Row():


            with gr.Column():


                resnet_prediction = gr.Textbox(

                    label="ResNet18 Prediction",

                    lines=3
                )


                resnet_probabilities = gr.Label(

                    num_top_classes=4,

                    label="ResNet18 Class Probabilities"
                )


            with gr.Column():


                resnet_gradcam = gr.Image(

                    label="ResNet18 Grad-CAM"
                )


    # --------------------------------------------------------
    # EfficientNet-B0
    # --------------------------------------------------------

    with gr.Column(
        visible=True
    ) as efficientnet_section:


        gr.Markdown(
            "## EfficientNet-B0"
        )


        with gr.Row():


            with gr.Column():


                efficientnet_prediction = gr.Textbox(

                    label="EfficientNet-B0 Prediction",

                    lines=3
                )


                efficientnet_probabilities = gr.Label(

                    num_top_classes=4,

                    label="EfficientNet-B0 Class Probabilities"
                )


            with gr.Column():


                efficientnet_gradcam = gr.Image(

                    label="EfficientNet-B0 Grad-CAM"
                )


    # ========================================================
    # Events — MUST stay inside gr.Blocks
    # ========================================================


    model_mode.change(

        fn=update_visibility,

        inputs=[
            model_mode
        ],

        outputs=[
            resnet_section,
            efficientnet_section,
            comparison_section
        ]
    )


    analyze_button.click(

        fn=analyze_mri,

        inputs=[
            image_input,
            model_mode
        ],

        outputs=[

            resnet_prediction,

            resnet_probabilities,

            resnet_gradcam,

            efficientnet_prediction,

            efficientnet_probabilities,

            efficientnet_gradcam,

            comparison_output
        ]
    )


    # --------------------------------------------------------
    # Image ID -> True Class preview
    #
    # This does NOT interact with the models.
    # --------------------------------------------------------

    image_id_input.change(

        fn=update_true_class,

        inputs=[
            image_id_input
        ],

        outputs=[
            true_class_display
        ]
    )


    # --------------------------------------------------------
    # Flag case
    # --------------------------------------------------------

    flag_button.click(

        fn=save_flagged_case,

        inputs=[

            image_input,

            image_id_input,

            model_mode,

            resnet_prediction,

            resnet_gradcam,

            efficientnet_prediction,

            efficientnet_gradcam,

            comparison_output
        ],

        outputs=[
            flag_status
        ]
    )


# ============================================================
# 18. Launch
# ============================================================

demo.launch()

Using device: cpu
ResNet18 loaded successfully.
EfficientNet-B0 loaded successfully.
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
